# 00 — Intro & sanity edit

**What you have:** a fresh environment with the model and the `student_utils` helpers.
**What you'll produce:** baseline answers, then a known PISCES edit (erase 'Harry Potter', from the paper) to confirm the editing machinery works end to end.
**What success looks like:** the Harry Potter answers change substantially after the edit, while unrelated answers stay almost identical.

When you're done here, go to the `01` notebook for your topic (`gaia/` = sycophancy, `itay/` = reliability).

**For every experiment, ask: Why are we doing it? What are we doing? What did we get?**

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
# Import PISCES + student_utils from the shared cluster install by default;
# override with the PISCES_ROOT env var, or fall back to walking up to editor.py.
PISCES_ROOT = os.environ.get('PISCES_ROOT', '/home/morg/students/yoavgurarieh/pisces_students')
if not os.path.exists(os.path.join(PISCES_ROOT, 'editor.py')):
    _d = os.getcwd()
    while not os.path.exists(os.path.join(_d, 'editor.py')) and _d != os.path.dirname(_d):
        _d = os.path.dirname(_d)
    PISCES_ROOT = _d
os.chdir(PISCES_ROOT); sys.path.insert(0, PISCES_ROOT)
print('PISCES root:', PISCES_ROOT)

## 1. Load the model

In [ ]:
from student_utils.model_loading import load_student_model, get_default_generation_config
model, tm = load_student_model()  # google/gemma-2-2b-it on cuda
print(get_default_generation_config())

## 2. Baseline answers

In [ ]:
from student_utils.generation import generate_one
for q in ['What is the capital of France?', "What are Harry Potter's parents' names?"]:
    print('Q:', q)
    print('A:', generate_one(tm, q, max_new_tokens=80))
    print('-' * 80)

## 3. Sanity edit: erase 'Harry Potter' (paper features)
We use the exact features and hyperparameters from the paper. If it works, the model loses Harry Potter knowledge while unrelated answers stay intact — confirming the edit works end to end. (This feature set is GIVEN; in your own notebooks you'll find features yourself.)

`sign = -1` means *suppress* (maps to PISCES `Feature(neg=True)`).

In [ ]:
hp_feature_set = {
    'name': 'harry_potter_demo',
    'description': 'Paper features for erasing the Harry Potter concept.',
    'features': [
        {'layer': 1,  'feature_id': 8965,  'sign': -1, 'why': 'paper'},
        {'layer': 1,  'feature_id': 13394, 'sign':  1, 'why': 'paper'},
        {'layer': 4,  'feature_id': 661,   'sign': -1, 'why': 'paper'},
        {'layer': 20, 'feature_id': 11104, 'sign': -1, 'why': 'paper'},
        {'layer': 20, 'feature_id': 14668, 'sign':  1, 'why': 'paper'},
    ],
}
edit_config = {'tau': 0.4, 'mu': 36, 'linscale': True, 'use_signs': False, 'description': 'HP demo'}

In [ ]:
from student_utils.pisces_adapter import temporary_pisces_edit
from student_utils.generation import generate_many, compare_generations_dataframe
hp_qs = ["What are Harry Potter's parents' names?",
         'What sport is played on broomsticks with Quaffles, Bludgers and a Snitch?']
control_qs = ['What is the capital of France?', "What's the distance to the moon?"]
prompts = hp_qs + control_qs
baseline = generate_many(tm, prompts, max_new_tokens=100)
with temporary_pisces_edit(model, hp_feature_set, edit_config):
    edited = generate_many(tm, prompts, max_new_tokens=100)
compare_generations_dataframe(prompts, baseline, edited)

Expected: the Harry Potter answers change substantially, the unrelated answers stay almost identical.
Note: the edit is reverted automatically when the `with` block ends (edits never accumulate across cells).

## 4. The helper library (`student_utils`)
`student_utils` holds the **plumbing you do NOT edit**: model loading, generation, dataset helpers, the PISCES edit wrapper, feature search, and reporting. Read it to understand it (every function has a docstring), but your research work happens **in these notebooks**: the parts you write (the scorers, the prompt/feature choices, and the contrastive ranking) are written inline in the `01` and `02` notebooks.

In [ ]:
import student_utils.datasets, student_utils.generation, student_utils.pisces_adapter
import student_utils.feature_search, student_utils.model_loading, student_utils.reporting
for m in [student_utils.model_loading, student_utils.generation, student_utils.datasets,
          student_utils.pisces_adapter, student_utils.feature_search, student_utils.reporting]:
    print(m.__name__, '->', [x for x in dir(m) if not x.startswith('_')][:12])

## 5. Next step
Go to the `01` notebook for your topic (`gaia/` = sycophancy, `itay/` = reliability), where you'll measure the behavior, then `02`, where you'll find and edit the features behind it.